# 02 — Baseline CNN

The baseline is a scratch three-block convolutional network with no pretrained weights, no augmentation, no class weights, and no label smoothing. It is trained for 20 epochs on the fixed 25% subset, with validation macro-F1 used for checkpoint selection.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
import yaml
print(yaml.safe_dump({
    "model": {"name": "baseline_cnn", "pretrained": False, "input_channels": 3},
    "training": {"epochs": 20, "batch_size": 16, "optimizer": "adam",
                 "lr": 1e-3, "augment": False, "class_weights_from_subset": False,
                 "label_smoothing": 0.0},
}, sort_keys=False))

To retrain from a shell (about 22–25 minutes per seed on the CPU host):

```powershell
python -m src.train --config configs/baseline.yaml --seed 42
python -m src.evaluate --checkpoint checkpoints/baseline_seed42.pt --splits val test
```

Existing results are loaded below; no training is started by this notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hist = pd.read_csv("results/baseline/seed42/history.csv")
hist[["epoch", "train_loss", "val_loss", "val_accuracy", "val_macro_f1"]].tail(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist.epoch, hist.train_loss, label="train")
axes[0].plot(hist.epoch, hist.val_loss, label="validation")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Cross-entropy"); axes[0].legend()
axes[1].plot(hist.epoch, hist.val_macro_f1, color="darkgreen")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation macro-F1")
fig.suptitle("Baseline CNN, seed 42")
fig.tight_layout()
plt.show()

In [ ]:
import json
for seed in (42, 43, 44):
    m = json.load(open(f"results/baseline/seed{seed}/metrics_test.json"))
    print(seed, {k: round(m[k], 4) for k in ("accuracy", "balanced_accuracy", "macro_f1", "macro_auroc", "ece")})

Test macro-F1: 0.5296, 0.5299, 0.6968 for seeds 42, 43, 44. The baseline's main failure is VIRUS recall; it often predicts BACTERIA instead.